# 2025-06-13. Sprint - data - processing

## Summary 

In this notebook, I'll prepare all the data for later analysis, so I don't need to repeat the data processing across each analysis.
It will also increase the consistency of my data. 

- Data is loaded from the Motus classification files.
- We assign the different taxonomic ranks.
- We merge the data with McLeish2024 metadata, to add Host taxon, Habitat, etc. 
- We save the data into a CSV file
- We merge the data with the Sanchis2021 Plant-Associated Bacteria metadata.
- We save the data into a CSV file

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
habitat_palette = {
    "Crop": "#0CAB7B", 
    "Edge": "#3A8DFA", 
    "Oak": "#FB2231",
    "Wasteland": "#FFC51F"
}
from skbio.diversity.alpha import shannon, chao1
import taxoniq
from scipy.stats import kruskal
from skbio.stats.ordination import pcoa
import tqdm
from scipy.stats import linregress
from skbio import TreeNode

## Load data from Motus

In [2]:
index_df = pd.read_csv("../results/2025-06-09.motus-g1/source.csv", sep=",", header=None, index_col=0, names=['library', 'read1', 'read2'])
index_df

,library,read1,read2
1,PV001,PV1_18807_ATCACG_read1.fastq.gz,PV1_18807_ATCACG_read2.fastq.gz
2,PV002,PV2_18808_TGACCA_read1.fastq.gz,PV2_18808_TGACCA_read2.fastq.gz
3,PV003,PV3_16716_GGCTAC_read1.fastq.gz,PV3_16716_GGCTAC_read2.fastq.gz
4,PV003bgi,FCH5VT5ALXX_L5_HKRDPLAgshTAACRAAPEI-203_1.fq.zip,FCH5VT5ALXX_L5_HKRDPLAgshTAACRAAPEI-203_2.fq.zip
5,PV004bgi,FCH5VT5ALXX_L5_HKRDPLAgshTAADRAAPEI-204_1.fq.zip,FCH5VT5ALXX_L5_HKRDPLAgshTAADRAAPEI-204_2.fq.zip
...,...,...,...
320,PV586,PV586_03661AAB_AGTTGC_read1.fastq.gz,PV586_03661AAB_AGTTGC_read2.fastq.gz
321,PV587,PV587_03662AAB_ACCTTC_read1.fastq.gz,PV587_03662AAB_ACCTTC_read2.fastq.gz
322,PV588,PV588_03663AAB_TCGCAT_read1.fastq.gz,PV588_03663AAB_TCGCAT_read2.fastq.gz
323,PV589,PV589_03664AAB_AGGACT_read1.fastq.gz,PV589_03664AAB_AGGACT_read2.fastq.gz


In [3]:
def load_motus_classification(x, library):
    try:
        u = pd.read_csv(x, sep="\t")
    except pd.errors.EmptyDataError:
        u = pd.DataFrame(data=[], columns=['mOTU', 'taxonomy', 'taxid', 'abundance'])
    u.columns = ['mOTU', 'taxonomy', 'taxid', 'abundance']
    u = u.dropna(subset=['taxid'])
    u['library'] = library
    return u

In [4]:
motus_df = pd.concat(
    index_df['library'].apply(
        lambda x: load_motus_classification(f"../results/2025-06-09.motus-g1/{x}.classification.csv", x)
    ).to_list()
)
motus_df['abundance'] = motus_df['abundance'].astype(float)
motus_df['log10_abundance'] = motus_df['abundance'].apply(np.log10)
motus_df

/tmp/ipykernel_1241046/61898022.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  motus_df = pd.concat(


,mOTU,taxonomy,taxid,abundance,library,log10_abundance
0,ref_mOTU_v31_06099,Mycoplasma genitalium,2097.0,1.0,PV001,0.000000
0,ref_mOTU_v31_03148,uncultured Clostridium sp.,59620.0,9.0,PV002,0.954243
1,ref_mOTU_v31_06099,Mycoplasma genitalium,2097.0,1.0,PV002,0.000000
2,ref_mOTU_v31_06426,Candidatus Phytoplasma solani,69896.0,6.0,PV002,0.778151
1,ref_mOTU_v31_01500,Neisseria sp. oral taxon 014,641148.0,3.0,PV003,0.477121
...,...,...,...,...,...,...
7,ref_mOTU_v31_10306,Paenibacillus sp. TI45-13ar,1886670.0,1.0,PV587,0.000000
0,ref_mOTU_v31_03148,uncultured Clostridium sp.,59620.0,9.0,PV588,0.954243
1,ref_mOTU_v31_05265,Rothia mucilaginosa [Rothia sp. HMSC071B01/Rot...,43675.0,1.0,PV588,0.000000
0,ref_mOTU_v31_01746,Exiguobacterium sibiricum,332410.0,1.0,PV589,0.000000


## Taxonomy ranks

In [5]:
def fault_tolerant_rank(x):
    try:
        return taxoniq.Taxon(x).rank.name
    except KeyError:
        return pd.NA
    
def obtain_taxonomic_level(x, rank_name):
    try:
        return [level.scientific_name for level in taxoniq.Taxon(x).ranked_lineage if level.rank.name == rank_name][0]
    except IndexError:
        return pd.NA
motus_df['taxid'] = motus_df['taxid'].astype(int)
motus_df['rank'] = motus_df.taxid.apply(fault_tolerant_rank)
motus_df = motus_df.dropna(subset=['rank']).copy()
motus_df['phylum'] = motus_df['taxid'].apply(lambda x: obtain_taxonomic_level(x, 'phylum'))
motus_df['class'] = motus_df['taxid'].apply(lambda x: obtain_taxonomic_level(x, 'class'))
motus_df['family'] = motus_df['taxid'].apply(lambda x: obtain_taxonomic_level(x, 'family'))
motus_df['order'] = motus_df['taxid'].apply(lambda x: obtain_taxonomic_level(x, 'order'))
motus_df['genus'] = motus_df['taxid'].apply(lambda x: obtain_taxonomic_level(x, 'genus'))
motus_df['species'] = motus_df['taxid'].apply(lambda x: taxoniq.Taxon(x).scientific_name)
motus_df

,mOTU,taxonomy,taxid,abundance,library,log10_abundance,rank,phylum,class,family,order,genus,species
0,ref_mOTU_v31_06099,Mycoplasma genitalium,2097,1.0,PV001,0.000000,species,Mycoplasmatota,<NA>,Mycoplasmoidaceae,Mycoplasmoidales,Mycoplasmoides,Mycoplasmoides genitalium
0,ref_mOTU_v31_03148,uncultured Clostridium sp.,59620,9.0,PV002,0.954243,species,Bacillota,Clostridia,Clostridiaceae,Eubacteriales,Clostridium,uncultured Clostridium sp.
1,ref_mOTU_v31_06099,Mycoplasma genitalium,2097,1.0,PV002,0.000000,species,Mycoplasmatota,<NA>,Mycoplasmoidaceae,Mycoplasmoidales,Mycoplasmoides,Mycoplasmoides genitalium
2,ref_mOTU_v31_06426,Candidatus Phytoplasma solani,69896,6.0,PV002,0.778151,species,Mycoplasmatota,Mollicutes,Acholeplasmataceae,Acholeplasmatales,Candidatus Phytoplasma,Candidatus Phytoplasma solani
1,ref_mOTU_v31_01500,Neisseria sp. oral taxon 014,641148,3.0,PV003,0.477121,species,Pseudomonadota,Betaproteobacteria,Neisseriaceae,Neisseriales,Neisseria,Neisseria sp. oral taxon 014
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5,ref_mOTU_v31_01209,Pseudomonas syringae,317,1.0,PV587,0.000000,species,Pseudomonadota,Gammaproteobacteria,Pseudomonadaceae,Pseudomonadales,Pseudomonas,Pseudomonas syringae
7,ref_mOTU_v31_10306,Paenibacillus sp. TI45-13ar,1886670,1.0,PV587,0.000000,species,Bacillota,Bacilli,Paenibacillaceae,Bacillales,Paenibacillus,Paenibacillus nuruki
0,ref_mOTU_v31_03148,uncultured Clostridium sp.,59620,9.0,PV588,0.954243,species,Bacillota,Clostridia,Clostridiaceae,Eubacteriales,Clostridium,uncultured Clostridium sp.
1,ref_mOTU_v31_05265,Rothia mucilaginosa [Rothia sp. HMSC071B01/Rot...,43675,1.0,PV588,0.000000,species,Actinomycetota,Actinomycetes,Micrococcaceae,Micrococcales,Rothia,Rothia mucilaginosa


## Metadata merge (McLeish2024)

In [6]:
mc24_table1 = pd.read_csv("../data/mcleish2024/nph20054-sup-0002-TablesS1.csv", sep=';')
mc24_table2 = pd.read_csv("../data/mcleish2024/nph20054-sup-0002-TablesS2.csv", sep=';')
mc24_table2 = mc24_table2.dropna(subset=['Library_code'])
mc24_table2['Collection_code'] = mc24_table2['Collection_code'].apply(lambda x: x.split("_")[0])
sample_reference = pd.merge(mc24_table1, mc24_table2, on='Collection_code').groupby(['Site_code', 'Collection_code', 'Library_code', 'Location', 'Host_taxon', 'Habitat', 'No_extracts', 'Longitude','Latitude'], as_index=False)['Date'].apply(lambda x: len(list(x)))
sample_reference

,Site_code,Collection_code,Library_code,Location,Host_taxon,Habitat,No_extracts,Longitude,Latitude,Date
0,C1,C1F,PV534,Aranjuez,Diplotaxis erucoides,Crop,3,-3.593308,40.051302,1
1,C1,C1F,PV535,Aranjuez,Brassica oleracea,Crop,17,-3.593308,40.051302,1
2,C1,C1F,PV538,Aranjuez,Brassica oleracea,Crop,8,-3.593308,40.051302,1
3,C1,C1F,PV540,Aranjuez,Picris echioides,Crop,1,-3.593308,40.051302,1
4,C1,C1F,PV544,Aranjuez,Sisymbrium runcinatum,Crop,4,-3.593308,40.051302,1
...,...,...,...,...,...,...,...,...,...,...
318,Z1,Z1V,PV590,Villaconejos,Zea mays,Crop,11,-3.476076,40.050052,1
319,Z2,Z2V,PV047,Villamanrique de Tajo,Zea mays,Crop,13,-3.131000,40.044720,1
320,Z2,Z2V,PV048,Villamanrique de Tajo,Desconocida 4,Crop,9,-3.131000,40.044720,1
321,Z2,Z2V,PV527,Villamanrique de Tajo,Convolvulus arvensis,Crop,4,-3.131000,40.044720,1


In [7]:
motus_df = pd.merge(motus_df, sample_reference, left_on='library', right_on='Library_code')

## Genome accession

Important to enable phylogentic-aware diversity calculations. We need to map each taxon-id to a genome accession present in the GTDB phylogenetic tree.

In [9]:
tree = TreeNode.read("../data/taxonomy/bac120.tree")
tree_tip_names = list(tree.subset())
tree_tip_names

['GB GCA 913047795.1',
 'GB GCA 023356515.1',
 'GB GCA 021803845.1',
 'GB GCA 902776365.1',
 'GB GCA 022775595.1',
 'RS GCF 000442275.2',
 'RS GCF 030813355.1',
 'GB GCA 001468865.1',
 'GB GCA 011525685.1',
 'GB GCA 031285105.1',
 'GB GCA 017433825.1',
 'RS GCF 900106765.1',
 'RS GCF 013839445.1',
 'GB GCA 018716805.1',
 'RS GCF 026625125.1',
 'GB GCA 003542575.1',
 'GB GCA 003280985.1',
 'GB GCA 903789595.1',
 'GB GCA 902782995.1',
 'GB GCA 021806885.1',
 'GB GCA 023436395.1',
 'GB GCA 021296435.1',
 'GB GCA 015060255.1',
 'RS GCF 001552415.1',
 'GB GCA 017524445.1',
 'RS GCF 001875665.1',
 'RS GCF 016908275.1',
 'RS GCF 947487805.1',
 'RS GCF 005871165.1',
 'GB GCA 913757435.1',
 'GB GCA 017619905.1',
 'RS GCF 017940385.1',
 'GB GCA 012839605.1',
 'RS GCF 003024235.1',
 'GB GCA 023266455.1',
 'GB GCA 017617725.1',
 'GB GCA 002425405.1',
 'GB GCA 018829775.1',
 'GB GCA 018623785.1',
 'GB GCA 013824325.1',
 'GB GCA 025054615.1',
 'GB GCA 900314345.1',
 'GB GCA 020627705.1',
 'GB GCA 02

In [10]:
gtdb_metadata = pd.read_csv("../data/taxonomy/bac120_metadata.tsv", sep="\t")

In [11]:
gtdb_metadata[gtdb_metadata['gtdb_genome_representative'].apply(lambda x: x.replace("_", " ")).isin(tree_tip_names)]

,accession,ambiguous_bases,checkm2_completeness,checkm2_contamination,checkm2_model,checkm_completeness,checkm_contamination,checkm_marker_count,checkm_marker_lineage,checkm_marker_set_count,...,ssu_silva_blast_align_len,ssu_silva_blast_bitscore,ssu_silva_blast_evalue,ssu_silva_blast_perc_identity,ssu_silva_blast_subject_id,ssu_silva_taxonomy,total_gap_length,trna_aa_count,trna_count,trna_selenocysteine_count
0,RS_GCF_000657795.2,0,100.00,0.14,Specific,99.53,0.00,426,o__Burkholderiales (UID4000),213,...,1528,2822,0,100,JHEP02000033.784.2325,Bacteria;Proteobacteria;Gammaproteobacteria;Bu...,0,19,55,1
1,RS_GCF_001072555.1,7,100.00,0.52,Specific,99.81,0.09,773,g__Staphylococcus (UID294),178,...,896,1655,0,100,CP030246.976624.978176,Bacteria;Firmicutes;Bacilli;Staphylococcales;S...,700,14,36,0
2,RS_GCF_003050715.1,0,100.00,0.04,Specific,99.60,0.22,769,g__Burkholderia (UID4006),248,...,1530,2826,0,100,CP012193.2238151.2239686,Bacteria;Proteobacteria;Gammaproteobacteria;Bu...,585,19,52,0
3,RS_GCF_016772635.1,0,100.00,0.16,Specific,100.00,0.04,1169,f__Enterobacteriaceae (UID5139),340,...,1538,2841,0,100,CP032396.2469808.2471361,Bacteria;Proteobacteria;Gammaproteobacteria;En...,0,19,86,1
4,GB_GCA_000615405.1,0,100.00,1.37,Specific,99.62,0.57,471,o__Lactobacillales (UID543),264,...,1545,2854,0,100,CP020604.1893382.1894929,Bacteria;Firmicutes;Bacilli;Lactobacillales;St...,0,17,45,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
584377,GB_GCA_949039885.1,0,98.92,0.35,General,98.79,3.22,420,f__Lachnospiraceae (UID1286),207,...,none,none,none,none,none,none,200,16,31,0
584378,GB_GCA_905234525.1,14,98.73,1.30,General,90.12,1.09,420,f__Lachnospiraceae (UID1286),207,...,none,none,none,none,none,none,1001,18,35,0
584379,GB_GCA_948663365.1,8,76.65,0.14,Specific,76.98,0.38,406,o__Bacteroidales (UID2617),265,...,none,none,none,none,none,none,184,17,26,0
584380,GB_GCA_948940555.1,0,99.99,0.11,Specific,97.55,0.13,406,o__Bacteroidales (UID2617),265,...,none,none,none,none,none,none,109,18,42,0


In [12]:
gtdb_accessions = gtdb_metadata[['ncbi_taxid', 'gtdb_genome_representative']].drop_duplicates('ncbi_taxid', keep='first').copy()
gtdb_accessions['gtdb_genome_representative'] = gtdb_accessions['gtdb_genome_representative'].apply(lambda x: x.replace("_", " "))
gtdb_accessions

,ncbi_taxid,gtdb_genome_representative
0,1331258,RS GCF 000657795.2
1,1282,RS GCF 006742205.1
2,2135698,RS GCF 000172415.1
3,90371,RS GCF 000006945.2
4,1236944,RS GCF 029023865.1
...,...,...
584333,1752224,GB GCA 001464955.1
584341,2665509,RS GCF 000006945.2
584345,2530375,RS GCF 004348725.1
584349,2169409,RS GCF 003097655.1


In [13]:

motus_df = pd.merge(motus_df, gtdb_accessions, how='left', left_on='taxid', right_on='ncbi_taxid')

## Save data

In [14]:
motus_df.columns

Index(['mOTU', 'taxonomy', 'taxid', 'abundance', 'library', 'log10_abundance',
       'rank', 'phylum', 'class', 'family', 'order', 'genus', 'species',
       'Site_code', 'Collection_code', 'Library_code', 'Location',
       'Host_taxon', 'Habitat', 'No_extracts', 'Longitude', 'Latitude', 'Date',
       'ncbi_taxid', 'gtdb_genome_representative'],
      dtype='object')

In [20]:
motus_df[[
    'taxonomy', 'taxid', 'gtdb_genome_representative', 'abundance', 'library', 'Site_code', 'Collection_code', 'Location', 'Longitude', 'Latitude', 'Host_taxon', 'Habitat', 'No_extracts', 'phylum', 'class', 'family',
    'order', 'genus', 'species'
]].to_csv("../results/2025-06-13.sprint/base.motus-hits.csv", sep=";", index=None)

## Merge with PAB database

In [21]:
sanchis21 = pd.read_csv("../data/sanchis21/sanchis21.tableS1.csv", sep='\t').drop_duplicates(subset='TaxId')
# taxid = sanchis21['taxid'].to_list()
sanchis21['pab_type'] = sanchis21.apply(lambda x: 'pathogen' if x['PAB-phyto'] == 'P' else pd.NA, axis=1)
sanchis21['pab_type'] = sanchis21.apply(lambda x: 'symbiont' if x['PAB-symb'] == 'S' else x['pab_type'], axis=1)
sanchis21['pab_type'] = sanchis21['pab_type'].fillna('unknown')

In [22]:
pab_motus_df = pd.merge(motus_df, sanchis21[['TaxId', 'pab_type']], left_on='taxid', right_on='TaxId', how='inner')

## Save data

In [23]:
pab_motus_df[[
    'taxonomy', 'taxid', 'gtdb_genome_representative', 'abundance', 'library', 'Site_code', 'Collection_code', 'Location', 'Longitude', 'Latitude',  'Host_taxon', 'Habitat', 'No_extracts', 'phylum', 'class', 'family',
    'order', 'genus', 'species', 'pab_type'
]].to_csv("../results/2025-06-13.sprint/base.motus-hits-pab.csv", sep=";", index=None)